In [1]:
from common import *
dfs = load_raw()

# 1. Missing values and duplicate rows
for name, df in dfs.items():
    miss = df.isna().sum()
    print(f"\n{name} | duplicate rows: {df.duplicated().sum()}")
    print(miss[miss > 0].to_string() if miss.any() else "no missing values")

# 2. Primary-key uniqueness
pks = {
    "customers": ["customer_id"], "orders": ["order_id"],
    "products": ["product_id"], "sellers": ["seller_id"],
    "order_items": ["order_id", "order_item_id"],
    "order_payments": ["order_id", "payment_sequential"],
    "order_reviews": ["review_id"],
    "geolocation": ["geolocation_zip_code_prefix"],
    "product_category_translation": ["product_category_name"],
}
for t, k in pks.items():
    print(f"{t:32s} {k} duplicate keys: {dfs[t].duplicated(subset=k).sum()}")

# 3. Orphan foreign keys
def orphans(child, col, parent, pcol):
    return (~dfs[child][col].isin(dfs[parent][pcol])).sum()

print("items->products", orphans("order_items", "product_id", "products", "product_id"))
print("items->sellers ", orphans("order_items", "seller_id", "sellers", "seller_id"))
print("items->orders  ", orphans("order_items", "order_id", "orders", "order_id"))
print("orders->cust   ", orphans("orders", "customer_id", "customers", "customer_id"))

# 4. Invalid / inconsistent values
print(dfs["orders"]["order_status"].value_counts())
print(dfs["order_payments"]["payment_type"].value_counts())
print("bad review scores:", (~dfs["order_reviews"]["review_score"].between(1, 5)).sum())
print("price <= 0:", (dfs["order_items"]["price"] <= 0).sum())
print("installments == 0:", (dfs["order_payments"]["payment_installments"] == 0).sum())
o = dfs["orders"].copy()
for c in o.columns[3:]:
    o[c] = pd.to_datetime(o[c], errors="coerce")
print("delivered before purchase:",
      (o["order_delivered_customer_date"] < o["order_purchase_timestamp"]).sum())

# 5. Outliers (IQR rule)
def iqr_outliers(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return ((s < q1 - 1.5*i) | (s > q3 + 1.5*i)).sum()

for t, c in [("order_items", "price"), ("order_items", "freight_value"),
             ("products", "product_weight_g"), ("order_payments", "payment_value")]:
    print(t, c, "outliers:", iqr_outliers(dfs[t][c].dropna()))


customers | duplicate rows: 0
no missing values

geolocation | duplicate rows: 261831
no missing values

order_items | duplicate rows: 0
no missing values

order_payments | duplicate rows: 0
no missing values

order_reviews | duplicate rows: 0
review_comment_title      87656
review_comment_message    58247

orders | duplicate rows: 0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965

products | duplicate rows: 0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2

sellers | duplicate rows: 0
no missing values

product_category_translation | duplicate rows: 0
no missing values
customers                        ['customer_id'] duplicate keys: 0
orders                           ['order_id'] duplicate keys: 0
p